# ML-07c — Sona : le recommandeur génératif mono-modèle, à l'échelle de la production

**Navigation** : [Index](README.md) | [<< ML-07b](ML-07b-SemanticIDs-Recommandation-Python.ipynb) | [Suivant >> ML-08-Clustering](ML-08-Clustering-Python.ipynb)

Ce carnet **digère** le *Sona Technical Report* (Yandex Music, [arXiv:2608.11015](https://arxiv.org/abs/2608.11015)) et relie son architecture aux quatre fils que le dépôt explore déjà en parallèle : **ML-07/07b** (recommandation + identifiants sémantiques), **Infer-15** (recommandation probabiliste), **RAG-02** (retrieval + reranking) et la **stack Qwen-Omni locale**.

> **Provenance** — la digestion est écrite depuis la copie archivée du papier au gisement partagé :
> `G:\Mon Drive\MyIA\IA\Bibliographie IA\MachineLearning\Agents and ML Engineering\2026 - Sona Team et al - Sona Technical Report - arXiv 2608.11015.pdf` (30 pages, vérifié à la première page).
> Comme ML-07b, ce carnet cite le paradigme et le vocabulaire du rapport ; **aucune valeur numérique** (métriques A/B, tailles de modèle) n'en est reprise ici.

## 1. Sona en un paragraphe

Sona est un **recommandeur génératif mono-modèle** servi en production sur Yandex Music (plateforme de test : *My Vibe* sur haut-parleurs intelligents — un flux 100 % recommandé, sans requête utilisateur). Un **seul** modèle, un transformer sur l'**historique d'événements chronologique** de l'utilisateur, remplace à lui seul les trois étages d'une cascade mature : **génération de candidats**, **pré-rank** et **rank** — un ensemble de modèles séparés consommant des centaines de features, dont des signaux de gros transformeurs (Argus, scoreurs target-attention). Sur le trafic réel, ce modèle unique a dépassé la cascade sur les métriques d'engagement clés (A/B en ligne).

L'idée centrale : ce que la factorisation matricielle de [ML-07](ML-07-Recommendation-Python.ipynb) et la tokenisation sémantique de [ML-07b](ML-07b-SemanticIDs-Recommandation-Python.ipynb) font à petite échelle, Sona le fait **en un seul backbone**, avec un coût d'encodeur payé **une fois** par requête et réutilisé pour générer *et* classer.

In [1]:
# L'idée "un seul encodeur, deux têtes" en miniature.
# Sona : l'encodeur traite l'historique une fois ; la meme representation
# alimente le decodeur (generation) ET le module de ranking.
import numpy as np

rng = np.random.default_rng(0)
history = rng.normal(size=(1, 40, 32))          # 40 evenements, embedding 32D
encoder_hidden = np.tanh(history.mean(axis=1))  # "coute d'encodeur" paye 1x, state partage

# deux têtes sur la meme mémoire d'encodeur :
gen_logits  = encoder_hidden @ rng.normal(size=(32, 8))   # tête generation (decodeur)
rank_scores = encoder_hidden @ rng.normal(size=(32, 8))   # tête ranking
print("representation partagee :", encoder_hidden.shape)
print("tete generation  :", gen_logits.shape, " tete ranking :", rank_scores.shape)

representation partagee : (1, 32)
tete generation  : (1, 8)  tete ranking : (1, 8)


## 2. L'architecture : un transformer, deux modules de sortie

Le rapport (§3) décompose Sona en cinq blocs :

| Bloc | Rôle | Résonance dépôt |
|---|---|---|
| **Semantic Tokenizer** (§3.1) | convertit chaque piste en une **séquence d'identifiants sémantiques (Semantic IDs)** courts, via **quantification résiduelle** partant d'un **Qwen2.5-Omni multimodal figé** (mélo-grammes + métadonnées textuelles) | **ML-07b** (RQ → Semantic IDs) + **stack Qwen-Omni locale** |
| **User History Representation** (§3.2) | encode l'historique chronologique, avec **compression d'historique** | — |
| **Encoder** (§3.3) | un transformer sur l'historique → **représentation partagée** | le « un seul backbone » |
| **Decoder** (§3.4) | **auto-régressif**, émet un **faisceau de tuples de Semantic IDs** (génération de candidats) | **ML-07** (générer des items) |
| **Ranking Module** (§3.5) | **score et ordonnance** les candidats en réutilisant les **états de l'encodeur partagé** (design **Gryphon**) | **RAG-02** (reranking) |

Trois principes de design guident l'ensemble (§2.1) : **(1)** un seul modèle pour génération *et* rank (la recette LLM — un modèle fin de bout en bout absorbe des fonctions autrefois séparées) ; **(2)** une représentation utilisateur unique, partagée à toutes les tâches (le coût d'encodeur est payé une fois) ; **(3)** pas de features manuelles — l'échelle + les données rivalisent avec le pipeline à features manuelles.

> **Au service** (serving) : encodeur → décodeur → **mapping déterministe SID→piste** (index catalogue) → module de ranking. Le **tokeniseur** et le **teacher ranker** ne servent qu'à l'**entraînement** : le teacher figé score les candidats générés et fournit des **cibles de distillation** au module de ranking ; les deux pertes rétro-propagent à travers l'encodeur partagé, si bien que sa représentation apprend de la génération *et* du ranking.

In [2]:
# Illustration du Semantic ID par quantification residuelle (RQ), comme en ML-07b.
# Une piste = un vecteur ; on le quantifie a plusieurs niveaux de codebooks
# pour obtenir un TUPLE COURT d'identifiants (le "Semantic ID"), pas un ID aleatoire.
import numpy as np

rng = np.random.default_rng(1)
codebook = rng.normal(size=(64, 16))                 # 64 centroids, dim 16
track = rng.normal(size=(1, 16))                     # une piste (2D pour le broadcast)

def rq_quantize(x, levels=3):
    ids, resid = [], x
    for _ in range(levels):
        d = ((codebook[None, :, :] - resid[:, None, :]) ** 2).sum(axis=2)
        ids.append(int(d.argmin()))
        resid = resid - codebook[ids[-1]]
    return tuple(ids)

sid = rq_quantize(track)
print("Semantic ID de la piste (tuple de 3 ids) :", sid)
print("→ un identifiant court et structuré, pas une cle opaque.")

Semantic ID de la piste (tuple de 3 ids) : (51, 58, 30)
→ un identifiant court et structuré, pas une cle opaque.


## 3. Le pipeline de service, lu comme un RAG : retrieval puis rerank

Lu par-dessus, le chemin de service de Sona est exactement la forme **retrieve → rerank** que [RAG-02](../../GenAI/RAG-et-Memoire-Semantique/02-Retrieval-Avance.ipynb) développe pour la génération augmentée par retrieval :

1. **Retrieval** — le décodeur émet un **faisceau de tuples de Semantic IDs** ; l'**index catalogue** mappe chaque tuple vers une ou plusieurs pistes. C'est une **récupération de candidats** (le faisceau joue le rôle de la query, l'index celui de la base).
2. **Rerank** — le **Ranking Module** **score et ré-ordonne** ces candidats en s'appuyant sur les états de l'encodeur. C'est un **reranking** de la liste brute.

La différence d'avec un RAG classique : la « query » n'est pas un prompt texte mais l'**histoire de l'utilisateur**, et le « générateur » n'écrit pas une réponse mais **décode des identifiants d'items**. Le squelette *récupérer large puis raffiner* est le même — d'où la résonance directe avec le reranking de RAG-02 (HyDE, cross-encoders).

In [3]:
# Le meme "retrieve puis rerank", en miniature, sur des candidats.
import numpy as np
rng = np.random.default_rng(2)

candidates = np.arange(20)                      # 20 pistes candidates (le faisceau)
retrieve_scores = rng.normal(size=(20,))        # 1er passage : score de recuperation
topk = np.argsort(-retrieve_scores)[:8]         # on garde un faisceau court
rerank_scores  = rng.normal(size=(topk.size,))  # 2e passage : le ranking module re-classe
final = topk[np.argsort(-rerank_scores)]
print("faisceau apres retrieval :", list(topk))
print("ordre final apres rerank :", list(final))

faisceau apres retrieval : [np.int64(4), np.int64(5), np.int64(10), np.int64(7), np.int64(16), np.int64(14), np.int64(8), np.int64(0)]
ordre final apres rerank : [np.int64(8), np.int64(4), np.int64(14), np.int64(7), np.int64(10), np.int64(5), np.int64(16), np.int64(0)]


## 4. Les résonances cross-séries — ce que Sona unifie

Le mandant de ce carnet (user, 05/10) : *« on a du filtrage collaboratif à différents endroits, on parle de reranking, de transformers, et Qwen omni a été testé plusieurs fois — il me semble qu'il y a des résonances. »* Sona les matérialise : c'est **un seul système** qui touche les quatre fils que le dépôt traite **séparément**.

| Fils du dépôt | Ce qu'il couvre | Ce que Sona en fait |
|---|---|---|
| **ML-07 / ML-07b** | recommandation par factorisation ; **Semantic IDs** par RQ | le **cœur** de Sona : génération de candidats par décodage auto-régressif de Semantic IDs |
| **Infer-15** | recommandation **probabiliste** (Infer.NET, inférence bayésienne) | le **contraste** : Sona est le versant **génératif profond** là où Infer-15 est le versant **probabiliste explicite** — même problème (recommander), deux philosophies (probabilités déclaratives vs. séquence décodée) |
| **RAG-02** | retrieval avancé : HyDE, **reranking**, évaluation | le **squelette retrieve→rerank** du pipeline de service (faisceau → index → ranking) |
| **stack Qwen-Omni locale** | le dépôt fait tourner **Qwen** en local (GenAI) | le **tokeniseur** de Sona est un **Qwen2.5-Omni figé** : la même famille de modèles que celle qu'on pilote localement, utilisée ici comme *extracteur de features* multimodal figé plutôt que comme générateur |

**La synthèse** : Sona montre qu'un **seul backbone** peut porter à la fois la génération de candidats (fils ML), le reranking (fils RAG) et s'appuyer sur un modèle de la famille Qwen comme tokeniseur (fils GenAI) — là où le dépôt explore ces trois capacités dans **trois séries distinctes**. [Infer-15](../../Probas/Infer/Infer-15-Recommenders.ipynb) reste le point d'appui pour la question que Sona ne pose pas : comment **quantifier l'incertitude** d'une recommandation, là où un modèle génératif ne produit qu'un faisceau ordonné.

In [4]:
# Le contraste ML (generatif) vs Infer (probabiliste) sur le MEME probleme.
# Cote "generatif" (Sona/ML) : on decode un faisceau ordonne, pas de proba marginale.
# Cote "probabiliste" (Infer-15) : on tient une distribution sur les notes.
import numpy as np
rng = np.random.default_rng(3)

# versant generatif : un faisceau d'items, du plus au moins probable (ordre = info)
beam = [12, 7, 3, 19, 5]
# versant probabiliste : une distribution de credibilite par item (ce que Infer tient)
dist = np.exp(-np.abs(np.arange(20) - 7) / 3.0); dist /= dist.sum()
print("faisceau generatif (ordre) :", beam)
print("probabilite marginale max (Infer) : item", int(dist.argmax()), "p =", round(float(dist.max()), 3))
print("→ le generatif donne un RANG, le probabiliste une CREDIBILITE.")

faisceau generatif (ordre) : [12, 7, 3, 19, 5]
probabilite marginale max (Infer) : item 7 p = 0.173
→ le generatif donne un RANG, le probabiliste une CREDIBILITE.


## 5. Synthèse

Sona condense en **un seul modèle** trois capacités que le dépôt cultive **en parallèle** dans trois séries : générer des candidats par **Semantic IDs** (ML-07b), les **réordonner** comme un RAG (RAG-02), et s'appuyer sur un **Qwen** comme tokeniseur multimodal figé (stack GenAI locale). Le pari de production — *un backbone, un coût d'encodeur payé une fois, génération et rank partagent la représentation* — est la version à grande échelle de l'intuition que ML-07b développe à la main.

Ce que ce carnet **n'aborde pas** (resté au rapport / aux carnets source) : les valeurs des expériences A/B, la recette d'entraînement détaillée, l'inférence d'infrastructure. Il ne fait que **relier le paradigme** aux fils existants du dépôt.

**À explorer ensuite** : transposer le « retrieve → rerank » de Sona à la stack locale (un faisceau d'embeddings Qwen + un rerank cross-encoder, sur un catalogue local) — un pont concret entre ML-07b, RAG-02 et la stack Qwen-Omni.

## Références

- **Sona** — équipe Yandex Music, *Sona Technical Report*, [arXiv:2608.11015](https://arxiv.org/abs/2608.11015). Copie archivée : `G:\Mon Drive\MyIA\IA\Bibliographie IA\MachineLearning\Agents and ML Engineering\2026 - Sona Team et al - Sona Technical Report - arXiv 2608.11015.pdf`.
- [ML-07-Recommendation-Python](ML-07-Recommendation-Python.ipynb) — recommandation par factorisation matricielle.
- [ML-07b-SemanticIDs-Recommandation-Python](ML-07b-SemanticIDs-Recommandation-Python.ipynb) — identifiants sémantiques par quantification résiduelle (cœur de Sona).
- [Infer-15-Recommenders](../../Probas/Infer/Infer-15-Recommenders.ipynb) — recommandation probabiliste (Infer.NET).
- [02-Retrieval-Avance](../../GenAI/RAG-et-Memoire-Semantique/02-Retrieval-Avance.ipynb) — retrieval avancé : HyDE, reranking, évaluation.
- [ML-08-Clustering](ML-08-Clustering-Python.ipynb) — suite de la série.

Part of #19341 · See #19341